# 粤语 dependency parsing：pseudo-label + Qwen3 correction + self-training

这个 notebook 是一次 `Run all` 的固定实验：

1. 安装并核验 Stanza/ELECTRA 运行环境；
2. 从旧项目的固定 manifest 重建 **同一** 803/101/100 train/dev/test；
3. 恢复旧粤语 parser，做小样本推理检查并重新评估 Baseline-0 dev；
4. 下载 `botisan-ai/cantonese-mandarin-translations` 的全部 `translation["yue"]`；
5. 用原 Stanza tokenizer/POS/lemma 流程处理无标注粤语，排除与固定 dev/test 的明确重合；
6. 用旧 parser 产生原始 pseudo CoNLL-U；
7. 用 Qwen3 只修订 HEAD/DEPREL，并在固定 dev 上做 blind correction 诊断；
8. A/B/C 都从同一个旧 checkpoint 独立初始化、使用同一 seeds 与更新预算；
9. dev-only 选择 checkpoint，最后统一运行固定 test；
10. 检查模型可重载，把全部生成结果和 A/B/C checkpoints 打成一个完整 ZIP 下载。

**重要披露：** 固定 test 过去已经用于四种普通话 Stanza parser 的模型族比较，因此不是从未查看的全新 test。这个 notebook 不会重新切分，也不会根据本轮 test 回头调参。

输入文件在配置 cell 一次上传到本次 Colab 会话。LoRA `.pt` 必须是完整 Stanza graph-parser checkpoint（包含 parsing head/vocab 和 `bert_lora`），不能只上传普通 PEFT adapter 文件夹。此版本不连接 Google Drive；Colab 会话结束后 `/content` 会被清空。

## 1. 安装兼容依赖

使用旧实验的 Stanza/Transformers/PEFT 版本。Qwen3-8B 默认采用 bitsandbytes NF4 4-bit。这里**不固定 pandas**，直接使用 Colab 已预装且与当前 Python 匹配的版本，避免 pip 因找不到旧版 wheel 而现场编译。安装输出保持可见；完成通常需要数分钟。若 Colab 显示必须重启 runtime，请重启一次，再重新 `Run all`。

In [ ]:
import platform, sys
print(f"Python {sys.version.split()[0]} | {platform.platform()}")

# Core packages first, then the larger CUDA-dependent wheel.  No -q: keep progress visible.
%pip install stanza==1.14.0 transformers==4.56.2 peft==0.17.1 huggingface-hub==0.34.4 datasets==4.0.0 accelerate==1.10.1
%pip install bitsandbytes==0.47.0


## 2. 集中配置与一次性上传

这个版本不挂载 Google Drive。运行此 cell 时一次选择旧 checkpoint 与旧 `result.json`/metadata；文件只保存在当前 Colab 会话的 `/content/yue_pseudolabel_self_training/input/`。如果已通过左侧 Files 面板放入该目录，可把 `UPLOAD_INPUT_FILES_NOW=False`。

- `MODEL_VARIANT="lora"`：必须提供精确 `OLD_HF_REVISION`，或让上传 metadata 唯一给出它，因为 LoRA checkpoint 不含 ELECTRA base weights。
- `MODEL_VARIANT="full"`：旧 full notebook 固定的 ELECTRA commit 可自动使用；checkpoint 本身仍必须含 `bert_model.*`。
- 默认一个 seed 是初步结果；可把 `SEEDS` 改成多个整数。
- A/B/C 都最多新增 4,000 optimizer updates、每 100 updates 看 dev、600 updates 无提升停止。B/C 使用同一 50:50 gold/pseudo update schedule；pseudo loss weight 固定为 1。

In [ ]:
from google.colab import files
from pathlib import Path
import os

MODEL_VARIANT = "lora"                 # "lora" or "full"
INPUT_DIR = "/content/yue_pseudolabel_self_training/input"
DRIVE_WORK_ROOT = "/content/yue_pseudolabel_self_training"  # legacy config key; local path here
UPLOAD_INPUT_FILES_NOW = True           # False only if files are already in INPUT_DIR
OLD_CHECKPOINT = ""                    # optional exact filename/path; blank = strict auto-discovery
OLD_HF_REVISION = ""                   # LoRA: exact 40-char SHA or supplied by uploaded metadata

UNLABELED_DATASET = "botisan-ai/cantonese-mandarin-translations"
DATASET_REVISION = ""                  # blank resolves once and records exact dataset commit
QWEN_MODEL = "Qwen/Qwen3-8B"
QWEN_REVISION = ""                     # blank resolves once and records exact model commit

SEEDS = [42]
MAX_UPDATES = 4000
EVAL_INTERVAL = 100
PATIENCE_UPDATES = 600
GOLD_UPDATE_FRACTION = 0.50             # B/C only; A is always 1.0 gold
PARSER_BATCH_SIZE = 900                 # Stanza token-budget setting retained from old training
UNLABELED_CHUNK_ROWS = 128
QWEN_MAX_NEW_TOKENS = 512
QWEN_RETRIES = 2
QWEN_PROGRESS_EVERY = 50
LOG_EVERY = 20

CFG = {k: v for k, v in dict(
    MODEL_VARIANT=MODEL_VARIANT, INPUT_DIR=INPUT_DIR, DRIVE_WORK_ROOT=DRIVE_WORK_ROOT,
    OLD_CHECKPOINT=OLD_CHECKPOINT, OLD_HF_REVISION=OLD_HF_REVISION,
    UNLABELED_DATASET=UNLABELED_DATASET, DATASET_REVISION=DATASET_REVISION,
    QWEN_MODEL=QWEN_MODEL, QWEN_REVISION=QWEN_REVISION, SEEDS=SEEDS,
    MAX_UPDATES=MAX_UPDATES, EVAL_INTERVAL=EVAL_INTERVAL, PATIENCE_UPDATES=PATIENCE_UPDATES,
    GOLD_UPDATE_FRACTION=GOLD_UPDATE_FRACTION, PARSER_BATCH_SIZE=PARSER_BATCH_SIZE,
    UNLABELED_CHUNK_ROWS=UNLABELED_CHUNK_ROWS, QWEN_MAX_NEW_TOKENS=QWEN_MAX_NEW_TOKENS,
    QWEN_RETRIES=QWEN_RETRIES, QWEN_PROGRESS_EVERY=QWEN_PROGRESS_EVERY,
    LOG_EVERY=LOG_EVERY).items()}

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_WORK_ROOT).mkdir(parents=True, exist_ok=True)
if UPLOAD_INPUT_FILES_NOW:
    previous_dir = os.getcwd()
    os.chdir(INPUT_DIR)
    try:
        uploaded = files.upload()
        print("Uploaded:", list(uploaded))
    finally:
        os.chdir(previous_dir)
print("Configuration ready. Input files currently present:")
for p in sorted(Path(INPUT_DIR).iterdir()): print(" -", p.name)


## 3. 数据、模型与代码完整性检查

下面的 cell 还原内嵌 runner。Runner 会检查：checkpoint 类型、LoRA/full 权重组成、ELECTRA revision、Stanza processor 哈希、原 UD 文件哈希、旧 split manifest 哈希，以及 train/dev/test 输出哈希。任何关键歧义都会在正式计算前停止，不会退回未经粤语训练的 base parser。

In [ ]:
import base64, importlib.util, sys
from pathlib import Path

RUNNER_PATH = Path('/content/yue_selftrain_runner.py')
RUNNER_PATH.write_bytes(base64.b64decode('...'))
spec = importlib.util.spec_from_file_location('yue_selftrain_runner', RUNNER_PATH)
yue_runner = importlib.util.module_from_spec(spec)
sys.modules['yue_selftrain_runner'] = yue_runner
spec.loader.exec_module(yue_runner)
print('Embedded runner SHA-256:', yue_runner.sha256_file(RUNNER_PATH))


## 4–10. Run-all pipeline

这个调用自动依次执行：Baseline-0 dev 与三句 smoke test → 全量 yue 处理 → 原 pseudo labels → Qwen3 correction 和 dev 诊断 → A/B/C 独立训练 → 统一 final test → 模型重载检查 → 导出。

断点策略：无标注预处理按 source-row chunk 落盘；pseudo parsing 按 chunk 落盘；Qwen 每句追加结构化日志；每组训练每次 dev evaluation 保存 latest model、optimizer、RNG 与历史。同一 Colab 会话内重跑可恢复；runtime 被回收或重置后，本地 `/content` 断点会丢失。改变模型、数据 revision 或配置会得到不同 run ID，避免误用缓存。

In [ ]:
FINAL = yue_runner.run(CFG)


## 下载完整实验包

自动下载一个完整 ZIP，包含固定数据划分、全部 pseudo-label 与 Qwen 修订输出、日志、评估、配置、代码、训练状态和 A/B/C checkpoints。可重新下载且已记录版本/哈希的 Hugging Face cache 与 Stanza resource cache 不收入 ZIP。请等浏览器下载完成后再关闭 runtime。

In [ ]:
from google.colab import files
print('Local run directory (temporary):', FINAL['run_dir'])
print('Complete results ZIP:', FINAL['complete_results_zip'])
files.download(FINAL['complete_results_zip'])
